# Kaggriculture X036: conditional milk-delay agent

This is a reproducible public release of our **older X036 agent**, deliberately published instead of our current private agent.

X036 starts from Kaito Fukami's Apache-2.0 Community V1 route, adds an observation-confirmed FEED/WATER retry layer, and makes one narrowly gated market change. At day 10, a frozen ridge model predicts the opponent's public milk-sale intent for days 10–15. If the prediction is at most `28.86780869330488`, the policy moves one existing `SELL MILK 3` order from day 11 hour 0 to hour 1. Otherwise it exactly follows the retry baseline.

The model uses only public observation fields at runtime. It was generated deterministically from 41 public episodes and was locally confirmed on pre-registered seeds 5–24. This is shared as a study baseline, not as a claim that it is the current strongest leaderboard agent.

The notebook reconstructs a clean source-identical X036 submission, verifies every source hash, shows the X036-specific code, and runs full 720-turn smoke games from both seats. The public archive intentionally omits macOS metadata sidecars from the historical upload.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import tarfile
from pathlib import Path

ARCHIVE_SHA256 = "a450925ffbb5080fb343bb35e8c70a45123031d06590378deb6274098a55fed1"
EXPECTED_FILES = {
    "main.py": "91559ffcc4fcacf69d36a060bb1c44fe86d327c428d83f91a12970b36ba0b4c8",
    "community_v1_1_base.py": "5f0d98a453866092fe78c9464845b7685c5b8ab9d4167a0459b683cba6b899cf",
    "community_v1_base.py": "9bdfbafb6755067182d88ce594fd46fb1d712713ffd6931e83d5d50e84bc6fb2",
    "x036_supply_model.py": "91b325430a0b82f953dd61837d0e520abf34d7327c69510e26a1846637d09c51",
    "PROVENANCE.md": "47839378094b3049f3d81740e792e5ed483f06bacaf6459ba93567924be4710e",
}
PAYLOAD_B85 = """ABzY8000000{`s&XLsVvvH%L7&+{w5giRQGD~E@ibI!~fK!5;&ND?{y_S-EXOtL+mbIxAxd++fF8%teX)m7D%T9n3|FnOD8|LcGA2LV46K`4L0&;4H-qfqVh`8th}=zr;u|N0O7Eae3!0hIss|IMG5mzOGnQ~&6C(saj9bmDaVI+OUW&{^=^*RddubbW5Y>t^Co)^(<no*<3+u5LQvUl&VhqLVq1*L8W3+rELRe*6H_jCIXsZwc=5&8AKm&!*Bs$4w^F1-B5U6ZyxF^ELI4D74>}3qe#NJ5y2QJIXi>G%#IG7QCe1pK*%;fY9z`!M{I#{8&iae}3q|&-rp^Ivy`4!eZN8p-r^e=44*S$-1WBR|s8?+kllf<>hQ9ZkyvNScNhwl?jHDKln|DpDlDTWn@T_rjm-H84cw_I+udl0v0NWV42o8iI=D1inmzbW}L)N7V21tu$lE0qX0kwJl~anA-{G1*1ag?amsp|5)YQ~0<VtMJ^6AqM<E*ufH`hi_9oBxP80G)B}1Hc6c$@vn##&9i+}lfMT+%(+Q%6I7nWcqWllJ|<sH5Cj+{4f%5{0^jo$zwR5*3?d7#3^ZxXIR!k_aBu4Mxkq3|AX=;7vtUDwF}WO?sYK9K!%fy!+w(79qXA1XvM$soXjF>ee-GYCVmIF1ku^CK7v#0s%Yx*7D<zyK8ak;%s*v9u3NT?}O*3`KxZNnh>#0{NpH%E!W2SEwJkk}qA172zcK4{*a*gi_lpv?BlL_AeqQFPid#pS}E}dqschUZ6Uyz?KF7hdEK^!KL$t1{?+#(_rtyYy6}uO!`-&+I(*c_AK1<=V8a63Lf-7famz5SqvpZsZcQw*LvRc7Rc!?-_rq+@$vxR2M8@zbti%i11-@1GHDtC3yGV^;O^L>?$@8XX<NQfA}_@S+~ZLH-<qB2Y<u3;=h3Qi$vk&83jSw-ljww=ZotXhVj-zJ2vDNjeX;6zK$y;g!Xd(BVL5~Lc?A-#0a#vM0(%3o>ZTI>?E>nif+1(DfHQC_JOfv__71F!?{eFZhJ#yRv@+qp|E<G_Yp}s|DSZS7?`eRhJ2*wAkSmIE|AOoOts3dxnRJ&9f#a2boGJzd=nW#clKxL1ed}2yH#6TT&*k<tLNH3X{X@Ci<D{_+E{THtWHHKJfb7?AWf|a)My|p+m+ofsbCd3wHE<d5zbo3Wv=V56tZ1AvM&u_J1&>wd2I72GROfwV?{I+19<Wo3vt@ca-)YX4ZBghbnygOrOPl4l)q2aJU)N+nPQ3qnKk%}#ZihjsT<qaHe=i&Q*9+WM9VjS%#feKWLl;b82b%i%Xbs<a^hNU5lvq`n%_Qaj_w9AHN7P$DWc#Bcvi(ujya9-0ZpNQS!i@p$Jpd@-(1VJ@p#UNkOD3+j(z^~{#C;*!dR7EgyKh|syrUp;yL6iMf0sO94Zz>!H^BZ#z&n9|bo^vF<|U}ysv-IXA_!^3F92`e0ngN^Kmh5lmqI9+e0e8a!}a?XMzIJWGP<AomqNwwhq0PUooy>rl%llcAI4qFdnyTiVZw<`7)^peXgpm#Dv0*emUwP-;jZOYML*^9mAk#)hi7Wvcg8a*Tz2LIn{`g;;+6R0oqPAA-nbdWWBHi#4;%B^A3;vS4?jNPwBpB}ZYd>1=esJ0<z;&;EUq@;y}iATL+)60O_E}>Kwz)ft@nkWebuuySb-7xSh+tX`jULyD+cs+e>lf-^5HBWa2&4e$B(N4@9su*8vN`!7*S<@jv2T<2=!GPq@e9DB_+&t!AWm}n5*zYwJaaOTGhDYi~&fbx!^^9408v|eKZ2Cv*n!^)*she`EI>sWhK9!`GTI}cA;4XW=8dwZ_t4Avc>xQ!*w@DT)(fWnJmNP1}sL}&Zfej^5eth|M;)}`}v>zd6FvskC%f!um7L~O;We@9~8sDpZ}}>{AcPv0Zj#TW%Lc?dTo$<bzzOCE%VY!*><m85E$_j-FgaKnWR&58l6(Hcmt{MfY&W&@&Y*Kv5p7(bvhP0I*^%wBuRCCYS;lbGYmy_z#vbTi=T>TnE;c=iQ2>xxX%9oIteD6bOzj{59m+|KMR24T%t;F!eXjhVdQ(AM+fy*6ej$et~dZbiQmk4Nf^UA%yc4d6-otO)^WOUC=~p61=z8?wu<!t%&AO26~*ZqxYFyDgSZj|BQh8OQa3Q_#~G?CbQRABivxgWWi4<La2rNwGBBHhDC(x_)LgH}!-AR)T;$(=e78~t^U43ks+j?>6dG)>7~qCF3t2Y?#$0%XwHe*J5eH0|0*5nz3IJAnUO5=7maEQFt%U_hcNQRK><JUdOsz)trt@jimIaWUfLq#<WQ9raGuH+vA-QiK*7I(wc^{SO47%jB!@;n1HkE<CNZYg8>6S8(LmQ|J4>Sy_C!k$f5X7JZ3tGr3@@6s<073b64eveL{G$9P|H9rw<A3X3E-QGszB+=M+yt6@bM>bDcdknPbr}u1VoB%#c|t9_W+beq-L3E%<Db?CuS#cMR0V%D^P#L3ffvBs%ny*XmO*fmE;cjDmHkUPlP^YLY+T5c)DIzFp$Pw0Lg4T5ss;wjU&o(Hb?W$4`3Q!tO5pNSH{!QHp>}eBC)be&j9O)30^Gr*!)t?KD0CCjE76H^QwfK!CsVlqt}bAP1wSYio%2jD;D4|T1)+uyD5wZE-l76MB2uqE!Sa6p8prLp^U|Lbqf|VcRzQFw-nu+LgCEXg)U5Xg+5TOvpc&_-mw(g>i3*{n2j@F#W#lF6D-=M`|MMLZ*z~Rp0DmcvYBqX3R0%mRneENgDyUX-`F%C%Im5b{^|P7I<L{=rl=>7KE???=(3wEv-BbuTGf(-fm54s3Qj;{y8dY$RD&9``E_CHzuyYnlmZ-5rSw|DDL4y9!atjUesrR|9zi$Ng>kp-)vnk-e{<)p{_wP&X6IvH;p%jvS>J)?eu0|I(0)I9?bz1RI8}<u<C~mHxSDEgaR*SMlRRk{(-))8M*Y@|XbFtxU@hve}e^4y`PhD|4<If+<a4RUEAGgyBuy=`2@I_mf62NGC10+5xVY&oCiuS45<=}pTzb==wHN3Oor{dJ%3PFSf9nHHvxt>U;cPaHju%3s>;JwRdFaQP?+MtI;C>bgIe>zi(aLO>i<;Rl9R7vJvV9fJ!Q`xvObSm+Rox2&LL@H`EWg-BleODWSB)$*n0i9V))#|WPgSZ_Gg>Sj==WJS3$VxQiD_)nc-MS}*6jhzog?zkDC9L~zEOaDHx-Uur{RQ}`bn)er6|X^3EInWKGs;!cZc=8ovY;vAj1#0M`R-38vTfej?ei(Z`<cKWo(4P|2K@0cz@Hre+*ZxMtd}dPfJ=os7vW(A3NFrhUH?{yKNn!NO#;#UddG7%8)xfwjS1McUl(a802k|}*wxFf6YflqYQ`Uk{{8#D*!*>wSyOzNfLwFuz1Vy;;_0~p<i^z)>zP3pZd_^H)iBMI?2q>@<QDksLaq^5&*kerH~#j~41{@pG6P4x--W7VJzK?h`p+2mY7G~j;K3O01tfLQbiAms-UlZ*5cbDaXam`;IrFP+1e4N@1^KCaU;}zS8i5o2y5Ri%3$Ee!2Z54oy<lANXCi)S)c^i*PDTRL$wHVcLE`eBUQhT}IPvSbLJ3n9@B!qC3ngR$Dd++wNt{l#4r)>YC8;b4xE)wBJV$Flv=?W;;@^Y?Kb9@mXJJ5MYHI3sDeKmVzb%}e!DrK%<&5M#Ue$g&=cT+G$3B0qgaQ^`5%uc@mR1%ozi)_ugMm$Y#0#kTr6B#S?UI|hfNJrOZ!cG|14#LMIt(fvoYo0Von3d6W!$0xJ8mjnWOR|QqK{|#(Vv%%_c?LOT<2@nclMrlCcJqzU0R&=Nxk|bwx4ayIa+vE>i12g`V!@+N+bIry10e7Tzv+j+Wfmcdarku_sHix=D9tG+6lAwo%K_9V_Kf*B!E>}@XzO(2Hh{*(;vj6*T<vEdamN5=A9HD1a>>@EW1mr^^~zU1%c<R{|eoe`FTWm9V9~w0GECfvOY!r^EpW{jJ>(@aN#k=y}kYPsuES*9x`e=@$<<-03Xh4(N4PoPqfv!?zY+&+F;Q6l*0)C!GWl<E6Bipc7P$(N_10RJ)KeuiSOmPf612=>}oQl_IMT5%Twi4W2^!yRi_0v<8>3@R&{XKz3EcZ6%UJy;2IB7G45(M&C1^yclCG)PQ0CT2GVpQ?0}7byW0}ingc`6+e`4-F?bwvT|0SZPu>mI{^1byY?}X3%=XX6+ZStllj3U>SZ^WT9rA;zR7LyjM6&mys`6eP0v7cqPS>zzbG$i~e1Ep)x{z@l72h0dont_VTO$VfeZ|%%HdzBzL%wTB^q=2X@xaB_eh~_&qu=f9<M8)ge+2+-EgTB|US0j*xY3}?z`8c4lLxRsdUj>XaV$_pSTqOwO4C7A_uk`&sn}Jfgj)1##BY_0@GtoP&NV{L!8=aPzu(sIUUk^-+t}^etwMjM;ku3_{`Rc&O6ZrzHa<Qmygpt1!~E*=Q&nZEhcngx{*mX)%S!@ihGw6ZoTzdG7P*yk3HWHlrB(<sBvCy}Xm8JTd*#X7zdSsBD6Ri5JPiK%`5%dr7<PC5htlN#o&Wu(&i~*^n(~C)D^Pe~QD&S3Oyv3C4tiY)9xq9Xl}KD4C!CM|fC|k-9{5frqi=JaQ5!mT0Uo#YgvG0J0;fKYrYlBE>0l^-`3pk)ArGC!_Qyr9!S3dCBCN0II!g(jrl^F?xCt-5>GDeu2LP|!1BMPFgNT5a71~PZDQ<s=!dOtw+NP?ze-)LJph_qd1UsYye72+>_$b9;m|`ie|CQeW%LV_1kLT!Qak`LyfISAc;I!8hIs%wv0v01ocn%ivgbojnM*t4>>3GfDo{82Kr)yv@_%T5B_9H!o=maHFQRW9!cO)J@+xAm;e&Xq;8Vt$8j_<06gy-`wxGyQ<*=gWVo=&vSV|c80{=5VH0H&##3`Kl_TGJOO#xiMm!U6%5GaA_f#{i!M#Rrh8DKbU8wHXp#?DA@Oss8>~=k;Ie2WjLqRR+^6Y0h_h)r!|2A0_Q$F18MM`-r}K++zBb6e9X7%Thlpv`0HJKeKgx^A)ni`Np&uA>wm$(6c5bZ)P-V%`^MrzE!YuyNwrX+04UwpRb)!jcr)`l`qPB$*`#JTVW~0A8ICpshIGJ&Lr)X_|PV~^;nF@RMVbs`__(~ujw^V!%(xH^ww9#=)CGS`m^Xhhf~$?M05@o_DyOKI|lH<-o<VTMQv3sPt%;uR}K8mea!cKQ7IN}3CH*@TPjid;?%iMGg&>=$dzRmVU8ayHpxu)xj?Jvm>shY#++nM!zIXi;@w#+#F%npUeBAHUh9<LJlR>1E@n!mGb*eZZRWya#k!!^8?VuHiVu&u4At@&NV_+Pa)NQ{SZz9zb?3`+LByRMZLINmz00_a>HL6kH^%OhM{4=!9!E4`;(RRTCz5QCc<rtcZ8;lXtK?i2yNQV>9NoVrXR#IPve;*tHEFE+rR8xdHd${1<?CWe+8cW5=4;A9cUsaZis}8~8tLH-CQO?xYejq$OL2p>xV@>pD)sDXRuIF&!W>AKjh5hAUqcQ&j+CjMHMbCOx6{KoxMp~}H}h>9Hkc$kb+qLv=3RogrX*`(vX3^poH1Of<Q*Ps-Qo5Z&ZrWx`pIgP^smMgY7)1la_Y1jg<IqfC(ABVIbZZO+u_%3*B=fnGPam}jE1_cb<BM{iR(ik$k_d-?Jh@#_a$3)($yAVT*}t8cM+EiMK@=wuuYq9i={JBpP?a5ZL`{t3`E<^YSY|JRut`%yfv@BF)(11m@Li0fkwi)we?EPEGh=)SkXz>15}gICg1rjZto83Ci~vTOwJ)$EL{%fH>rHl<8=C0nelRG%XyD>nxtEJbYa+rWGnKTnOhu_ayZvm74uk<u&wH3Ca*`h1=T@unki+BKKn{TVbQn8i`8K@F!wU`Lu<UZk@$WeHRiPe^0?E&Ee(kYEp0o*8J(=qK4*$Fh!vJIHjGqha9SeeWto!lYg@P;=gcK<j%+6KGp;^$(?JjIi?ThrEezXXn+eSh2HP&S9LxS;Y(E#uXnh|`oKcx3L3DiV)>nMEKO9c1y<WK!pY9W<5o0h}TE#$dRwMltTpudr?c>RK<UItc1y`nI_na_~%#7S(#uv(&{bItr@$zhcKMhF3O(z>?kOfM0myEA@Ms*w&>9r#{FO8y()nVeSHTrEMN@FoO+u_1i6W`kGtZi=4U3wDXidUS>GqZ5GQyDnNuHKlcuc^2}>Q~3;aNQC7PEK&{l8%!o8oSd9kzE_&)S_}m<!xD#rp$7vF;pLDq1IAx7?g^0WXc;2xY#U<E5^UKaa3fq>xJCr?%27RHe8WiIW5oQTe&5)TziTTs}7<S5rSksjtf|2>6E<WVI(&6Mc;wS_*gp{*RXW$BPs3j;z(?nL8}c?ptL)|ZiMd|L=zt!2Zt7#um#hRuv3;+Ya`Y<1uR<|>JE6vs}vFhUOHSV2XUd7-FS!sT_-%%15y+`xRdhsQahsDn|iq<KX(lq)$%ah=_CwmVp$GoM8pGJhsIF-G7@X*aXpu`_)O%SBx~70vyjFN+hxw5oYuNLT3Y$GseyPb=?nUkMrv#2n7Bflg4aGKv*CrAjf=GvOH_J(Q=)Hcx7nhB;$3c3Y9{pb)^?iI({UTYV!gQEUboB$?}0ICb6G3c-sPMTIKJ{uxP5<>&9;Lhxt>OMbdZx{jkGPH4`4nl5F4#h#nzD@I9}(5<s!!&9KLqylqAB@<aXSOSjPLx-mr3Rijv9b%jW}$S{}34hOfCZDj&Cs?$$<gs$g#bf9+V3yLB!pdYj#rGab?h;+bY^((iRz7|8{gRK)7`q06-+F^neBjpP}IDwM<L==Be2n;4SJGS1S7S7;yRJi!ZLgFX{B`K<=j7%x(JtBX5z`Z9Ym3}fCYvT*DY%fx9P)7IW6$Jl0`S%rPZNtDY)m{}<l3%ct4e9pVvd3XMX$2T=l*6_67&efU|qpj^f_QdWWLOJVt!M6_=ZQHc%K-V%Yx-3mYq7W`kVnL&xp4QXpPA9FoAkQ@8$x`9TNZ)x~PPfLm)$nRdmPseFS#8BEMpU;Mswq&O8aDNAaGTG=^SG}f)HCE1AH{p~xMMz=VK#1@eihja>($sr7QM_;3eQS1&9OByHny`k3vwypHQilR+pAEehOOC>C!gGho$3C?enq463Cj*wLf;xeTk+k-$2N@lt#co*CQOcPX<0Rt{Ap^O_ZC;%yo-&PO(wV1&P06qS9UWU3*v+cbYFupL)oHt53N;y;8b^%DuwxUE{Qd%CAehGRBAX8cI;f1_=D)GAcg5>(MkHzVZ!N(J2xb{tYj-xw%)OZw`o^9K1`kJ=A@ls(sVgP1ZH$4xRWT(U0-*tk*29`vlkOqYBsR4yVFi^&dB<_mNNQ^8qRXMD$<sqjWOFLXSEUI+sPDf*^!GN8#y$>mV&*vYqcb!yVctc^s)Y+f=0UsJC6A~Q+tiDocQv@Y^OQHR*d;endxlA6bnOPx!U`31;J2v?ep5Iq>UU6QiUzz4txFEE?C`!UNtr;S8q^0jCL7c+i9}XzG=W`BrpCEi=WV*f8i%awn;e}_aW18Z#DAG-02uQJ*1fJ=vZBP@>J3;q9Iq$S01~q&P~NSogJDrXDb*FiTY4{fOzD_<Ula&W}Xz>rrCt#gxa*VIA-+J8CB)58=+1&y_u)8<{VRZ_Jf{6q~@L<(xqLUquWLL^@Oqgk>J@Q+r~J})>3OUKA)}yQ>E*lt>^~Yrw<N|^vr2*PxkB#oAC)9y4eksiwmkDdzbxO@6}(VD+$)<7j0{sxjHbXU#n!_6l9`h?v!=47AK)lkqTI6)=#|daZ|J4nric|y9Fn!r-f>JT5<)lL?PFoxaG(YTa6Bp9_tQZ`?1+R9WK3-m^E(3Qd|=$hpU*+AjfCei0FFz=Gop@%yRf@;Om;Jtym!#S(t_$Zn91KkPRVb&!}<%1V8#c6gOv6Y;zjRhc0nk%vUH+XQa(B$XJ4&slQ72gOP%M%`oCHY>Wi-SiX_xtBbwK?Z9`QgoyLJuSYtQj#bB%Em?PkaC_X3lY^E$mL1rAlGC{gzK*g<yR9Z$eVsnJwwzHF8-YDzXxL_Ye(6VNxvjT4^)wA0KV}=nNI#wFRZXj~Bcwmz)ewI48gp{q8AXcC6jvY}HoraFDy|K_;35$_*d0_amhhH0i@Z~hdBZ6pv)S`m!x>e7G}&f~L(l4%t7aD!|L8R8X)6xn`|6~03duX?cvNu(%KEjlY|^_9{h3X$wNhknuw-d#YtWBN)n#8God$6p^-XN4P`O5gGrTR?wCpIUVhU{$B9UX$?WKJ)p6rUH<1$k|RVudr8C8De>u$P6Cf>qnbeF}D;ii{odokO{#)Kv)#am2FPU^-Y+{|m*zFssA+XN4(c6HOka^1MeR!Gm|iA;Xg+BK_{Vd%u$OPy7pagq9JZDCmv%`$f^M}qw5bv-gqmi}CIk`Gi+b1Tz6RSTm~CK1?CjrE|rq3Xkf-&Tu7GG|n5s^~C<3N`&MXt$So4f}RiYUF(NB9lz4EImBHx|0qeQXTGUY?|`Vdf8er(5=NPom7sWTB<8H>KJvr^}HPoIksa2EB27Um=uB)zib{Ev$c(Hc1Fb{=OZ5$>r$<F)YfH&C8m_@Wn$5y$3`7m`{FE9&=}zB+06E3;IOP^rK9W{5yjvkR`xXP@!XM!uf2%FldFu7e5qCm$HT3yyXMTqz1hgTI`f@K##~INvJJ03*X`M5Lz|BY`%>B8b@hj@p>(O$$__eaE!Z65b{q3L>6`q`buKU-r2?H3odAZNh^->sxGn4oTWVE5>BGyTN?nk<8(Y3^I8`=nN3WHh5e-9u->owU*)gFZYeHK=d6z*WVHf9(3tKitmc0=?60BkUJ?~vvi-(D9&iaD!xTR?D?kWz^ddU*UU2bNqy2DK$8W@$$dH*!AV!Mmk<QTH{dxUlFakm_NmfU6!7GdHXqhdQ+r=nf6p4!k(HikAmd74RP!nUEoAxq;Tvrn`db<&p;S~lKos_H$OmQ5^?hSS-_Iz(yUM}mQFAV^1yS+;@}Xs&ASq<8vky1pnGhHUyk$w#4CWRZl;LOXi)X=aL9c^Smwm9WKiXj^)Qtu4a07Fz?~@U32Bp%f)q7|s3^OyOp*w&~Fmq{LvQvRO!kqoI*SFKt^DYsXBl62)o6XqM~FV9z#7829XCVH||#lw5O362a6RiL}ic^NkwmOC;y-nz~z-4RWbKKhMN@*I|%Ca}&D1+p#kuQ*NF*QgUA~q{#e`iXX!XNut8(b-NTj9vr~(E}OyKr0vM>HdxhWSg`(@DH+WSgHcZGnb8FoT${Ch94H~pm9rGuu2S1cGtr;F2Dfc?=_qv~9b464_oVXV%4jnVmWh0Blw8jW0y7MA4J%)YW?eDM3Tx@Hx@hX-yGbn8<+?H7Y&Z`O61j*W9TZmGo%4)}dhLr5Q+={ckJ{5F;Vb#DjYk%j<@)S+3Ed0&ZoY7wrIQvReG(7Hh;O@h7nhM{XxO(UrL}8th?gv}SZdSC&zk<@cHFEU#I>DFWV17R+bkfn?y1M1qgZlk38s>rU=mr6xc!(YRL~}68sxQXnP{b3fq+9qq{DJOPwd^<TJOYEgl5;1-HEAQ;;`fEmblxKn+1aLVn^bg+qBbXiYK~p_jD%AQb^{a+Xr#FaI9s&vEMZ%k|QdS)CZicN~@aSeAF}+4!OiIxk#Atoq1sLdYo*t)uQka8v~v?6n5lUY%@t`ExYC_s}D%zB2%j{c>`T+iW-W2yj2y<q{R{qj$wYj%rlH!bU7mJHd$pl;V2ep*WGdeC-z?dCYkmn)+IUV7-x!H_0)D$#%1})^-og4As#ZWR5y-t)N*tQVuqn`Wj4a{bGq#(CZ$|y&3j0r*BiHYtMWWF<y?-6zgp{#Dxro$4A~`bX66ATw;G~nK9lDQVrY#qE0bIZ%)>sj&1K*VMBVK(&*F;%-axnB-5J#(qu+Mt`h9g|lpBTxMm3BDOVyf3Y|(9@R@}c<Jo%1$n(n&Dgjbk#vf&~zY!`<4>1yTN9R_VHJ)&lmTO)mpoJi|f+7O{q(CeBY6`Yd>F6;Uf4EKFwzb#af>f#K+Rw2u>A8kxZkrv{~2UEofm)xT^u2`?EeL=)m8`?0hnDDJCJ+s}F>DtU~OS`b!x|~}r{<5trQs>a`cg_28GS@a&`Zbf-?r?M^Y%&bmi(Vu<^CrBO4C+~R`$=MWY>M$}q)#u|?80xZXSX8hX^N>~yp@bIe17FLlvBJZ8YBY=gR@|)(pK_}Ds9-fhUm_m4UdIk(vT4PRhwK+4O?}xl%66+J(kFjOGmL_b(D&j(6^^^#Q@@U`qQYLaa2si%sO@*XTi{GXGvolA}5lpml&E7HG1t9@>bHDM$f25g-!qU7!-m*b0NBG52QfO6}39`*4I!dV6HD-Em$$wkC?16$8I-By9UxcIPki$WvNuiyBMoolf)uiv3U)_?#9H8u)<vGSP(tgFI#OTlhfGPre$6~a}TzyA{Qc8fkqH-N(+A&<h};VsSkQ{;n%`?nCd5!>*Xpvbe*JJ0gcrp{hE;No<Ve9wNQRd5`DK=!1vA=;+REBaVrq@XydUs+k<?&gxGmkQtTQGdLOsg7I=+5XUf@(E_R_9Qs7SW#gg!yV%rnmThtN5ZpD_fHlLXsOmp1=T^j2hvG~Y0Ix*#w;}8`uC0ue6Ec-%n#MJPc<c;jIdz**N(VJtLb&0fzk#@PT=w&u8JC}9!@FvzuheyQ&8ct$Sim+TZoOuxq9a7F>RQ7e<&CDngwMrmIYfF~4f%5hEaMzyzu|^Zbw1Hfdv+*rYgGa=8rrWaVtubS_Y+Eg{GGPl_Xjju~uDI({vd_EvwdKSv%_1xN7T2PaT4UDF%Jke;=~@KC(u+-!-UJ%UPCKDkIUAaWa-L{bqR3>Sn4K1AOTHH=RFau-tI;&{{C-|e`q$MXmO~u+rLSd0>`Zh!88PV0vlQya;5fC@w6@$|@79GZWeM+EU`uv3M!oY`5emH%J<e__MX|*Q?&@$7U5X&_F!qw1V6;VMIe9@jTN%DoOwOc|FTvZ$v478`Q=7OxX$$hr5GDzHXl62+GAS*PqL3cpbqCtgoHf@s8=gm&#TA7adn=;GHq+BMnduo^s~lgQ+m5b4&*GYGioo`|@`0_1(KqpxC!sfXXfcA9F2c>F>ibG!zVJ4dp2d6~-N*-Tkn=7qwfV$T&f%BBjlCYvMCOt~j&i~@HV*Aa+sTXzNBme*G<zj)*5^04rEY#>(<7xaK{)M0da6I6PGKBj0(>@+9wRx2r-Zk=-k3b(QsFuu?I-pIY;V|a@IfwMET2(DtYh~G6`gs<INtH~+U6FSG#<8*zR;Bn`^+{uHx8m>p*UodD>@HUtt3u1`S7HWOVbwGH<Tg-vaBTW!7(`-zlH`qBnBe&*V%kOneVw_vUa?bAhJthYVM2WfCE8jf4*k4${A!ou}6H7!vJ%p>QQriKkhb#lVzX8F(P`2YyF4Vbg>+cEz8+z8^q@|GfwT1;j$J92c;!3c5Yj^bSy>Yj(NeBHyhh@jX9#Wb>APb&7^W#Pn8R!R<C91N^MgNYx)8uqdi?r%W^1U?8aC7Lr^P1gv%8o#E*jq<CI;FMPpCTR~>oV&`fPxO-tsGHU<tMyBD!HQnW^1dt^v&_-e=~MGMVx&^p_o<bH)(qhg?EOJ#dSa^GX^c8u6W3-~&^9|e!jmci0zyB<qs#M)C9u@|2h{8(O)BZDJvqIau#GM9}`1YEi#H*|u`><M8T@C4VBYWYNTLU`6-8{xYMwJ(?l)^?KLIqj+PniP_CIo2_y7P(ruJtM}OMJKHvWMdW6JY5-FN~w!hy^-0Nz%FFy2+5vWcr`GeL{Gc%BvWedM35<#l8q1_mtP}HYoecfHxd>}Hdp0U2<>|Yeuv(IRr~a|UrtEr0=p6y72@Lb{dSXOM;z>4efF?(cv2H$?m*`&Z;SWZ3h^N|ug``RvhwP!7PeD^fRFWy(KIW)#yhhv+h&iJ{aUEiwiY>HeD&`+!}4H{G+SiRe6+RMM6ox}FdZU38R;ja-EQqbgTmI1g_<HKhGlM8j7fYN(Wkn_RiRq*Zd1;<yt|YT(^L(|^J!N;nsa8vTnsm6F_vgz{kG?@mIG^PFv)sn@^n`(lpNdix=$uY1~2cF#%rNov&KUXw`li9sb!DOX>)FAiVtSX3^B7~Ufdk^C#Vxzux{$11a0R>BFM?8NXUURo)m{1S}58j4z-;xB~b-)tyjsIi>ADBH<@j^jnde6s;?TWd@Hc+TcQ>Fu($0sCz13S6<2T1GFyt?;vi>SSUZlIf%XQlOm=`UsX*ai1i3R-qdUdXcxazF^5(d=yFDIbrEZ<v>78RgzA=R8GRiV>R`xd~vnQ1md-l+ZPRxz%W7Hkw=4Vu3-rTY%)c57RtD2r-RU)(}%VTHEo*`1%*PVY%F!-=qw6uB+kIUVWNqd@(?J7PHT6lxw0!Z`y#rU)-U>?3&iEQ%R)LZwW`IRf15BJd0y1FN{$k|$623w(8zP?IP?%Yhzi0i4b80&7zp3t&d5oWdg%sI-3Hlo{t6kR=Q%Dh^$?M>c|qSetTh01t-#+m)=m8*_#n3HpayG@M(x!AnQv#nq1*?ebI13tDE_R+{B=nMp`o|7fLB1(tQYbKD9n>nwqQl@vBTx*HR+8>%Y?WB7$?*|gGjKMLU7N|MJ_|b#I>lZya$9@MKyF6%ELJ^+ma&6k~*%h&c(U`31IUCiDBKGbmn;NzH*<HhwOuUv;q4hj$O!bV(aBO=<W$?utjoi4W7n;j-U=P2mSKf@=dwbOCs+vt^U%=I1mXSP%HM97_C>9&;{<x49vZcyMpR+X9q?;=>N2F2elabxD;68CK(@01!o8y{qwKPv7Z7rA8vw3TVDK^#<M-<CU8p6!ex1X}xoF~)5%I3Do)S1j&Qj!b>qi#<swCcCIhpap6su6zLWVe&ji4MES6kbmuex&QuC&v|wVJCMJnXIu?B>3J+3#t%`ma3&=ZDp$jLnb7IhvR&>(2p;=RDla;BB|GMx1F{mY&_0xq?mO<3O(1})(%nR32{Yf&pNe8gT${{4V8~p%3xVYdV!B8K)gKb^npV?F1p&eiLb<w!AJ-pGpX6iP@VdWF~=ddI0k|-vS~PwX{#eso*o?_!u9TZrAtwd4uwZKQ66@(xY08AC0(O#$UCZZFc;&mnng<Hok_{-qg?h~jYvh3AVuvQUG^xt%^OmViB)zPoP;;*JG9q^UZ=R>%&kN%H=}Z^;I6VXc$2w?R(MX$3CB$CH`z|uh?7yXH|L7f(p=xhw7XGzSYHbC%>GK(x@^Uc7HqPPkNf6+(X^gNt*<^iP0P7jBiO;Rm2OH{RjU0w-I7JFQ*vK|K%DnTBc>LaO%qadoM_d1!Nh7kb7X1>zn3Al9X?>+c~4G*KkE`IZKTHfExkzl&{~Nlva$(9Sj3km?Q_f>^q#UIr#SCzBM4=rrKA{Zj&0R}RWlEReddH!YkSGMz&X<<o%Umu{w|!%#kVLS#V4+~vxE%dn?ZS9i}`bbekWyNtN9&rT2*R(I;O{EBirajy0ghR*7B5gk-%$wRx9<R8%N*KFt5VF<QbJHhMc2HBg1o(Y!)vwHbz+Jy}M8(J{`S+NIqdNk18^G2-<}#X`Augdb=0dud)e!IY>?`mpapNVydL=$4WH0V)nIUQe2L!jeUnrgt<x}m5UqKqp6Tu7{+>w*a~)}QF2Gks?@AeIJ&&2uHNXYI<p}nsSgLtG`(Ln)|=yaGUrBd^VS+4Wel8F3doN}z1}W{zuNV2&QWZiT={LTwPK@E!gQ3QG`sC>c)>ObY#lL;yh24Zmh!Gr!xKA=x7=vi==(}Mvf_DZhSg2Jlc!6q-FUvAS+9F8-`=%K;-Ok})iUN?VnrGq*FJuo9}e83t}l|SR-7J7mXnG>N5Q!iG8U^)XhzSd;@dD{@`p#}Y(IdR7X?GF5HlGnThZ)4HcG{VRGD|T2?1}TUXfjPMq^7fYoVu`=x(ofQq5CIjKy5GR(RvX{A9P)K$0SbBeb(k2OCvSYjUbsG!q~e#6qf$BNl!uzM{m)T5fC>NA#%21GBzd6AO$X*D=n7%@9OO`L_L%`&GRAL$q~Va=x|c=xw?Sv>+C}IdPPhdcM7%H|tG8*<?0*TZa-`#^{|dfX-x+boLQUA~N%xe#J~fBM(_fqs2@M!I+|XkPZ4{F*>n9>{ZTW4r-{DHdn1F$C+fW<~t$_+(nO~B3~#cqB(JyvGh#M{K3#~MC2WVt?>q1&Mn-hTsq_r<K<D+i;?}oup#gulZ#?mii`I03APz>t;6eMvtSP=%Pv8S@^>O$-kIn5X>pTvRM2L)Se$UkOb+%L(l=dIvc!mS`1~u9#w%XVf~WU3AxIQsu^#KrrB;qg#pT=+`EfYvoTF2#@8CuaNSc_Dqhtp`$RTgm@|nUK*_B*FkFn`VpmuMr5bSIlovznE3kuaiB|Aym*9lAZ<eOCjAz{`WBuG;)8(S92!HW#~^*UFp7#r0tVJPEU3TqTXwxBCEN@a$N#L+(Dj6H)LnG6Kqsk{@YC`y`FX^VetUtWS(p%z<u-Bh87j$JNm6{}<uQGIqKyzYoH@Guj8wK&?XAZTtghP>Xp9TkIhmeI2_d$lZPhFeo~y)uR3dCcM<;x%7E?_F<Rhy1=<s{~6RE0-Mujl0VtZmst$GR;(SXv0!#ELtX{H5dx|tz$k|7TLaU7Ie^tNUE6F9`!D6wdnZL*32L`+a*BUkU(=`d|5zD?diU}?gbA$*5DZu#FVYnE$L&RE{IwVWwYCf*We+L4x^o#XH#H(tiNVhIOIX*cmTm@Mb7ob*S^14tZw>iyiDb`#(*KU>Y{~ZxGN1D({*r%6GEt;t|6Q(wMN-_G-kAy`wiD@U|hB?b-HoDcyQMX27JZVYF@cNtNFNN?qEyi8*OX6a?o!VLl@t*4#}05%FWz;gQZRethu;7l_(^8?n<;c@H6^!9IJ_jLBr?}#*IoZInrBG26UL%6QhIXpyorLIc6<6seW^rvyRF;3a8w-X&mrRyP-ge?Rc`MlJ}IaGqp^%6e&?TD;g^GY&*UXBj*9c<2ARPuEMBPs>do=D#7JwQ_EDf8j}1-&s+4H*~zGtP68{hKU*YU^+}f)Aq3R643mf9aaZuQS4Ddg+coL=!qPnS0yJf_4-;Ecc0E|P2}bY6!=6_e_2g2+bhnTSG68p%OLzp%Z%XOYl^9#oSQkH>=3cusGewi3B-@OpD62VJpx2n#EvNgrg|$0bu13_>8lbw^D>dpEZNtIQ(Z2-tgHT{n^tj3~Y2weFDgleRzL*#1nFc~vHlsaNPqqv4d{W6t#n~}3LTdbGJ01weqWpTujfIBI(`spx8CDC-n{x&m{Zt@qG><A?Y+MH^g8l^3S(2-vv4J0#it*}XYg^{}Nk1Mm(Niuado4}-j_9D)NjcaOW+S;R)&1M4)j<guVvzlLEF$!}J$rPjl@$p`&%_j5)-lLSrV-yUjD!!-_`sR0v?AjqUSDmHuIPPjyUiV!GrldAkFpWX1q<f3<Mr~;%djG5ghoEJh^6$iQ$jyRT}%cm+l_9wFWjlesOEUI5CX@3CKh%f;)?b)=3aTlnZ(7>laE=7L#BC24Bdm`)(HZMcC}jJY|PBMHV68t=_yi*o8}8?K@G9S+`Mb*UB&GQa~CR-%~YPDBxlCSc@QAr>TZ|ACFf^$qbO$!AuMAq8^T4KF&bK@cC$vAJEIC=tytmM2<r6*7ddpsa%7{qo{q+sy24&=Ehz7<m5uT;Uht25{jwnz6(*wS%oo;4TgAnv_o2|%U`gYhLt!Hu-6<1mWvq=(H6eDqhES(94G}pKNrmj$VaS-}1KvWV)kBS)k{)g6U7DM1Tx4$+`q(aHIm&&#C*8%WtEPV$+J{d}z?KVyPLeQ2PmTbdC5H1f9?CL}^_mG%)|l~_E^ddt(>AoC+(Ilr?9xUD!>6~qjxSfiNq?LdO`DoubY-fSqw@VUzMJF4tth%nzRN*~zaHvVvBqq<D|@X@D>@OUSgAu~%PqPtWa2D33-2Ps<zx)zF3vkPpCOud_m&(dS`uP5oCaIHV`9aVVNIWsX@5Eu8TO6Xx_E3QI1DH%-C?F$az0y~8mJ{@>DKC`Ff5E~dE<U;GMK6}iyYhwMl5OSPd5Fa5O8@)$(Fe*4D;(A%B@N{**qZ<En@;$N87UPwt8wqCRQ0qPN^l*6?C{g4%+<O6F#Hrc^A#i%9_o95Ut}sSxd|?5n!85rb6*kHW~|=Sd6Z%EP^8!banTrh*b_Vmp65cnG=f)J~VE;J7Z&Gisc8yB4wwUjbI*TCkbDU2nU>9eWxO7kLd*FBVW}D;+>P=HrFbFe5%nxi<dm2DCSN5&A=549B_8pi?Ig3&@(y$?9p8}*k^+jGDjkXrc)X>P)|*eB4pqg-Rb#2THo!>N8PB^T4@G{!2-8(i=pD$J|qH$Q4m9R)uM=VDQcL|9*+wpxXh|6pS(wLKE}!;ve+H|>)b8IYJEFDFZ{Ax^j?`4s}**7@%xg#bRcx~=<(gd$UmCdd<+Z-Co`Em{671SW-yfUrGrghG8^@QE65v0{%EFr`9!GL%=+@aRN?YvrgP(fe{>d%{Ha6X_Xd9I2>9zoo=dRP3i8H)f5d5D`HY>vv-A@1<?|T@5WfN>k?XP7lh7JqF_ZEYGZ2tME`XSVQGFy_8BScoQW|WbnD<rup?tmu;Zf{;0`!(-`g%6>eS%adnMtdweTT)7N2<X`{2eS5i9n!h7#w>f_<>1EV@gQr)lyOHhA-3ylvXmCq(Ufiy-4yop%;YVP`(&T#_FNGf)@FL!`J^>mFCffXeJ93DibUPidS0V_a#Cq(-mI))bvKfn~l}$J_T4Ab8g6~$@*cqKVM1*qRm29)u7usf8NMh1F8A)>1Lsr$%hmOg*`vDn^6A)`H(O5PQuj`?;S$#sRsKY;0voWo)!8sXCalz6oDQ3bjl0gefd-<UpQ|vl|pSX+QirFxLqLNJN#C~hb@_un-5K&{;wsI#P@|UZ@K+))_KXWZxw*OZoP5$jkTA>bUg|1aA1KK#n>4FZbMAzu>k&DQ}iP7ZBB$Nd^xuaQ8Bn(I*d@i5BVbUeXg^8`M;h~f7>(a(#-wl+k)Tk(W14x2P{#?!->z6ezU>KnT9`Zu%dOp5{3Rm?Ev+p)&2X|FF$H>uYI*zr#6~U=)79bJEyE4w^{k0=&tp*eYKu;)Y7_8ynjdd&y?}5vDWXK4nXRp|8H-y1zSq;Y8#eYb&=IZLO<1S>0K5IMqhmedn`;Wuw9GVP@11v;lBBK)51h)-17Un;f~g90d{iI%+5WTT8EJi|A`U0QO2vm4{H14er8qh&$~aIRrThZ=ojCx=K1dV`^p--lzpRZ#T(u|Y|4)|@6GEcE<5n1$IrVb|FY4K?!z8GPa64Lznw#s_jPZdt7bm;1ZW+7zWg%$xs%kr(R}O}^`u!8>=5<PNb1olsoa2Hl)h@i^6QBc`+3t>?+g3&`|H+0f8Fl~Zd{l{J>3ne?!i5!?meCTsuQ~6vLN)iRnYaS|9_-k&^1Zt7ZLn7n+83Es(*s;OWUS46jFKoxBY*v-uQ6Sc}HE@=Z$x6M0X{-vqHS<!$s!GMn7u@{Qvb<G4A^HXw3?*qI8Y0pNcrFml5a7U-Wc2$ER>4?AxJ2f~3S>iY0?hu(66IkVyU18uVHJhZ+U}w0Fz6>J|u>@X#bs5Bm+mc7$J0@X(6ow)4tkr<IFf-!0>F<@@y#j72mvp(*@T`;!|ewFk<L{d#PK@)+dY%%OEd(VA*}=xel)c)-ng%LEv;pK5w`mX`7vzRUu4VS5AFz4`(hb;u>An<)Onb;<RgXM7K%<@pCAb+!*7pg8Vf;Fsy&;H@)Vk7P~zA3%XS=<E|P6!`=Yv7m1VFi2P65+?vjy@y;1OD|kj^OC00j=%!_z+)#;_{IL8I)1(sM4>GSOE5$Rc`mrt=Edo{*;`IJ+Gn7gK+S>8YEM`E`HuQoy>rv^>b5rb3u6C$)k9j@%v#e==QROWmS{W^g~hT972=u#ATFil(!Vu5x+Ybj{<__lpu~&{fc<Jp+cV%RGJth^=hcr#+k8#j4HNE(yXMws#JxTv?i%==U$57M-XwHz-(1&D)wo}R-6t#xHl=-|svUw1qCHVOr?)q`>AM)O%$_n1I2ve*K*X_}iM;i;T`G(S&dIVHzLW0yo^l=98mQ^YOKzUrM$g@=u2tu(G#b%LQ;nwFnM%AG&s~#$w$)FUcAfSrM&;#InDovq?4Wuofrn(#D6u{!SMwzYO!#7}87P&zDgdUvJ4;KC{xBs1R!)6?l-g6}rfcZAuJBRQ(5*IF9e33%bo&6K2w#2xP%l4>KDQ}pUSqnHD&Cv8=~SDdRAaBZy=vg|<y2{es`C5YG@Lt}0?gVJ&rH<=5o(uE<jRUDonY?J{Lxgv6llV<-d04EIWG6WPQ19H+f}_@BfqGp+R~*t12?+d_GwcoNRgJ%!Y(j35{Flwyo+o)yRGfOVPN!fB2GIa_yQw1;;~c$S0ro1d|EYx1GBsoR9g6+Tla!L7xc}m+9PDhMV9E#&uftd0DM=8RA)a=e(oo%uuVDEK`wzU#0$-VvbgK_+fBPm_#z6}d-l4rueRz0gMp2*-o$Up?iMB&(A&7|jNA^q*Db^TeY<q143>v>c$ijX(k8#I+1<XOOj9&i;8puIJ}LG7xTaa}x;_55OxICLyv=j*Sf^BLmHvt9+^(C#u1X4a5B!lxRk7anZCpIA{CoB1TZ*j@vjKtM4!LFBvxbqMbfNydhxKEBYQ+M>7n=YNtQ-g2vE%+#c~2RAGe~_oC!xLi5f;?X$rE;U4Lx5&t<To?v0~ndP_a`N@^5<5DRXG%NpV%z&ht)u*Zxu&1=aK;m#b;&b3Imc_eu%jhE`^Dp36ml;;0pf&mHt1`Q-m)=lb7tmlx}&x*B+B{l|{vJ?4<*>%8eRj|UTXg$A{71)|nTN8z6^ff)ucpXd4heI^ZT7y$8x+;~b=n!qHgNy+U2y-l0;{alNij1*ZnmF{V`h0ABBU0<IjT%`I?O1nFdQOTaJmBQRb$1H!i9e(Ow&o!2-(;({~zI9Wj<i;D_DZSb|{`$BAQVLHiu!sG?y}x-C4tkTPAPJTD6)(xW3%*KI%L*qW<&Z^y$B|$p*a-@q;G=S=0TJA!_1b$VA%kEOTZ=k{^$ESI!BUrcbDp8j@$o6fpS1D33Af*?skhen@d=W9^ZbEH`pR72KZWvWlFzS5&`utZuRP7O<A}TUK@ERW*uSApJso@82eR6J-g;O1f7&Jbu`u(UJ$ZFha87VS${%p=>IJk9#rCTcy?at_8xOx90c&Na+X&?Sp!e~N?yLIz-AwPNyH|IP|B3fg{{PtvDmMsD0Y=Pb;ud}dh!_9k{a<Mm!S3Jxh*B8(fBj$oL;YVP{6z6%UF`*nRRl-WxAaOvx6kVoy->CyxI?|^f>R~18z^r9><SB||L`|nXQ(tP{gEsAf_YzBd0NZzlCHq2ep;#R7nIM1(2MnlavZ3RI3Fv+H^QZ|#hUs`IQ4>h7yutx%4D+2h4Vpy^+zDDzFF?ipKJQ5J#A<G;Y-I-zN88s$z%#46^BY*G!)9dTtXiZ47B;nQ$O0XOfa5$;`A;(&95up_qOvq2zWZBI4sEUeTtWNe4kzPbsqK2qLc=zF5vvW_#QCjKz{Ps(sucdpx8$cpSn=s3ENXn>Na7ATIFA;0;*qeVaWaKhwq;Kc^d;O3g6(O@k+U<hH}as>-94~%FTfQCOf}k=^5Ph{U^}SO8|$uoIppQJ@9gSfN!yCaL3D0vXUFV54{}hTCC4n&A)Awi0Ya+wFY_)^wbN=d#zNFy`oR2**=>FeKHMtJB{{q!qc_g9w5Fy>+&u)>vhqEmto&x*Ln2y!wNs{-vR+fsgF3E&!g10ohk5YR{V<oywG^H+@3<-+g<@XzMWx|wX2)qeB~>^TdrR}2J;2r_Bo)IYHH;+n1`MN4wb8yIvmKbx`2xdO?@sr)^0#HUyR#7kK~97Iojys)!MIT(D0qBeDGnN8!2m#?_5iH`!3=4jd~?vIA-(3n5$z`1%>)h5R~ustP#p(t5VV5Tweks&sOR32+&=~uFmVa-2dmHLZrTI-1U40<>X$O<NWfFPvBJ_@d*C>i1*`(U@_wE@b}_@tk#G0;(HVUUGE9-DUF^Y&bR&c__p61ZwEo4bpW%4?)g3Js!x2xbWyOqI`8pK!Tq+sKws<HnJ(A++;@*{NXq*K-`5T=yjXP~^6<$EY}A7v`1Zh?nY0%QetWJ5Rw&;6PdEXI|9}zdLO<lg1I}|$@F!T+g?@lnW6Xz`;S03k%Q)2%=sWwauKxuO)pz7Po3!)Sb3U$Ip|J_FBxS+pH2ON4P>x=KwSQlWRl$3`XhD5_kMf$Jt5+C)rti|Sqr{(ogt>EvxbtUtp+@usUhFD)>EpX|w(wpPJ~{#I^TV4WUqHer!&heuD;|tG-+p=W5}qD>1Dpuo2GE4>JP@1b33?yT)_Xovhr6NRGO^U;R-B+b@D8ufc%X=reiIIFwx^S&d^ef(PqcooFz0yVXD^DT+_QeEV%izevsc9cJaA<59J0bhMe)<xQ_|-U7QX!JNl<%J-;AY;p?n~eE$Zw#c{#RdBNbBBCs$wyuRS}Wi2mw<7R3Z*Lg8>M5DNike|Zk?$B*}!T{;YV!w43mX$;3`oFGy9$2HU?RQ}g{nEbDQbguw2LgF}1Vi?X66b5F%exASM4nv4^NF9aZILb04&f*l!kTgS`2Pr%Ule|4uH~<Uofr(KHApmU@MuQyv0TkLzB^^@6<1EEsIE|tpX$M%zr^DZKDV073V_AkGQHG=_il*_WKuUIYUJp&NG{YbaLXs4N()Z(^ZHqI%7?eR+iojSDWx=lpPF&2=dG!oN0D-VH&LSv_V-M?59jM~DpJI4ij0&M7q$-ZDNV>8XDgeu36w4Acf&i{D^xg8&PoxQyjHIAAS%#$%mL?xSZ>5P6fF6ti%0tm8`F!-fFi8fd83ag_CRhZp?<vrgFcB1?!P0RGWf+#k2#UFf_-k>Z_u@ox3@9{EOo~7lntT}kU1_4<l_vTqO#-D!mP9cOrAPvPh3*Cb`)qGA2#tV!fKxcI7=V8dpx=-t_BYbRo=6iVSsEx8&`ug>XokLr_*j}K2^a?L48l-ALhQqO9;8W8(6ZqeFdG<H-Tn9nao%DeDdsuEyEwlj=~0|GG?pk!;K1tQ_W<~Z;sn%T6wXip5!jmyg+2lOj;Y1}#?<1Erj|v4{l_qjW&jqF!tMe7TAcX3I0=kIFq{DRNCfym;(qvdrHOx6n)rh>p;^Kh0&Gy=1AsHQ10X(==2v!&xR)maS5DFdh2S(vkQDnI;2Wls_{?;|IDo}auwhZ46a-7$!`_JpL1_$FK;V2(2I?n%KmNPMlXx_q2#euR|1lgOLx5I4hk#XGRX#U?4TIhff7qQsu?S#oS%v^S2J5*q&*bw>gF=6SvcSv|6b>8*@f1oE3>Z%VtBJBKqz)K$oVg$WiCCaOFakqa1W1P>aPl7Z-Uu=zAPmS4g+3ibEBE8SyKOWGwZ8HPY$hM&_~jxV1Op;hppFa;V1Uk`_pX`x`=Exp-Dj`p8#FQ?1ORvt5+m8C!*5goP>cfvBtZa$rh%_vp2A%1OrRw=Ln0K(5(LmW=3zM>YXHVFD1iaN0Jjc;f=AF>1wav&0Hzt34Pd0eHpA|RKggd%2#_@pAlV>k76j7I0sdJ0)PwlHT*0mM0ZG7$0q25|1kDm8!rV!a{;OR{-|xyV^o0J*m7yR&10Ee^P++7P9K8p<+msZ=qR>4PfL|<5-;aI}31k*Uk|YLVITSd-4<P<nCiJ80`)UObt`C?Jm|+2~5e!M(yFTWxWWsz?Cd}W<g!z_Cm~Wd&=DRXszAF>v*4uq;RhaJzk^P=&WS`3%>{sBiL1KktD0D>6k-W>C0BfdzpJz!1prG$_YWDAC&VEbg?6>zJ`(2r{-<3K0C_)nW9WXzLpFl`K-cW>oQo2z#BTFDaTfkl<LBMkxQKOHw8z2n?mY-n!iGc+@9sZq!0)1Ay0l6nlfmj!Enz_x{(7&(Upbxbh6hx9B9>Osk2bm~_Jy7<&coqF_=>~mPx`8Px5QUn!0|L=o2=r5Nk}M1G0X~CZ3B-Na1MHnRK|;kqFAp+O21LF1)A4tK4oU(4ix9xJfCPa6Y1mVsD|uod`(|JQ0yaKQvj}q!@QFZilBNMRkSPO;KmfJ4hrQd*AjbzwVOWAf0cpg;g1#$J^t%#$TD}2U8_+bsHn0Ttj$hbk0{wjJJd|&M6#(`h#8wyu3@wUKPse|!fP?;hIR<?!$H0OI2;@ju&Vn(({qXO3SoFKq81z{+h6Ypt5rH_A0_x1(ZJCFX%ANhcM}U7X&PV^gJc2%!M}Q9psR!697+^4nu<nO{m%aFR*^56>ewHPHX{8vFL}69>7J$%>Hv^f<ACW}-H_YE%IYluqN*;=WWQU}I_;KPX#GL{BG&(__6k)!k>k|V=kN}VYp&8gTGzg-I4`Hv);r@`7k4JS{;Rk`sVrZ~T6j;<pz`FtiMd84BfW;9E4tB?bkX1E;57zyOS-Vty)CchXa549)dY8kpkK?b-dH-<T4_PVj)*!D25d=*Afkxc#Ir6`uuH-{;o}z)#1VJ7KEIR>=&(rbWioMbM%GB2e>+efb=woS$pfO;pfM*24VsPYP_;)SVH!D-<w<}ZVv&s~K&_IYN2p&L0d2iyVPwV+0oCe7)1>y;UL6Aq#XA=GF5CFZeGy!=6>x<Cv#04W^#Ct#d192+B7qC1e#72Omgu(A2?}Q0V00k6{K?xGU8RCJi?+TOpt}v-@3zK?GrdSYq6BG@M2gpF^yJU*~bUy<#2#g>_A%HmIK9~BiL<P_x7@&~?8vy9RQ>Z_cCjGZnL-e6S1td=3Ks~{N7+4&@?uUP?YKVTfLWMr5P<^?A?+KIn{A`8<*$R%)3<I__{@~P@=ef~mxjXvz#VYi%ScQTN2bL{xC?yc{5%=T2SGqvITdqRCTdqQ3o#&I?Lo*;FKrjR(eBjr8a=`wLE&N_#3H|%R68faD1Vc&&@|r||V<7H_e^)&0cg4egUp)8w2S!0)EK7i-4Q#&K=mvX3Zlm`v*o)nllaw_h1Oc`(&EhPEV<i3*;5!K`c3)8XqJoTlUQhzDH_pN@7t!Do&@kdd*xT^?4|wr3`i75S0RKrS?1pZUdBAwBXG#u3&?vCDN>m8;4|YHPdv5m~;ywKSgT>sc8jK<d7)As83lai~LFh+-+wl7j7XBE06BN*Auw;gY53S(0p%?aft?aWL1iLSleXXj`Yh@VRoGgXX6b<V!>;r+Hs2&g!aK8{M$Sgo;@c{dv2q*!v5sU=s1gxE6_v62-2<Z3Hf9#?5rHTk)7_b76RWSsG(|4=D{yIUz?#o|LM!+coZD3(l66^|wdJ1qSBN*j^{FQ<s7{)Lp@f7G!(|PREf*CL*6a$SsaBv8$*4%^M%M{=N*(FO72r$Le18LtCDfV5FVoyX$VmR2;VBat_d{hp-g}^nZpnSRQkEp^Q1gSh5!JzOw3dHaC`x<|~*FHD%_@`!`g^?71fwly=Zu~>o?^RhK=!aDnnj(NtVNf7pphWlf4u8-Epa?V!`w$xbq8`?Fs|;TnZT#D17VM!S_@$x8pVnEvTKS{Sfdm@_g(L!>V1f@a-9kK;5$`rN@xgdaM**{d^Q0LJmVYQ3SgU)0hezZ!%>ju&tRIuWB(T8lK2Y|c+MktIu+NKu7z1*05R<U*VIH8WAH&{9BM1Ys85RSEm<4m(R>QE*N`W{mJOa+(@T3&@y$8U1Cv>ws@&VCfY;g-gA@~O%S1R`v3m=0E1P8G)$O1v8M56RVl=Q5$N3sa4xe~w(fiMBTQ*81+_C;}M*<rj$5MVU%!SIV!1nv?TDgrn~7R2J@Jw@b48ux|wA-`X;#=ctw#6B$oepaElfqXQjpVdMj=m#M)0s<w5f<+#9wgi~+u)Mp{21de12jMX$4UZ%rhJV+sezWZO4sc_8Kg(sYCzZ;t82w}yAn@645_l1iK#}-e(Exj>Z7M4#P!@!-7z-Io6L-7w`Ll4Jm))>WA8z{^_Ri#>C<UL@2e}X27Z~v{{=3qozAMe|)*Z3W>xiFcCiHWTlQ_)+{|l@GPzx4ko&x>(2B$wM3j?!A!KW|@cuK+mZ1<q|J^^@mVAugefP<y(zrcV!D<uLDz;2*<&>X-D3jQ4Ak3~d3dXcZ!@vs@bU>frz^!*a#+hs)T@5_icpbzHvOHw}y68@j|&hNKz8;Rrh?7xCQAMR!2N}S;j^=;9nxI_2aqAl82%eLGE8?OV~n{FQNfA4pcq*;>`Z3?&p4krSO7P}ftq=v(pk7mg8f}tnNaJb=(!1^LKu0!|C7f)GVst-P8fY0`$2&g>?M9%M-n~Z0qI>|`oyA14!wT1-d5uP*@%Gly_qT!1*22K{@Fo#K7a~Mh|pl5BJc=QO1NwQY_fW*b`HHMzqK2Wx@G3zx^5@-5KOuv;34BU&70wwuz6ywZXq(Z|qv-jnx>Vqh_OLcWxcSQ-Zh;;-WFiVmbiPsqklrd~d<i}-a(5t{j<s6h|gpzn71rYH0Oh6TBUdNGRZhlg>c2YiPlhguXse`fNIuZ@ad8954k%@5x|H^7gULQ{9TuC1#((keX17#UxHreS!v}`dE(NT*DC9r2z%hyRfWg}@GcmvU}m#rqR5|=Lylz^pY2QNiNa^W*20jtz@!{W;43HdnM1PZQ8aU5j}O3#=?ogjI5Au<ROKR9ZV*O!ny3W;pK%#nt9Y+q9(g{YW-h>M&b7A&KFB07rtQGx&)SZtW=suH{uWRKD-(7hPqQ7Hs|5-5Xslmy2PH(~3fQWAFx%^ihAMO$a3Y?nNiEz$#5_j{^cvXCahbdESsREv8d_M<#7&TX|oL%>+dxIFAzgMg*RYC+*tqM|H2O5oo}f@6tT4Ww#7DzmgapRP<O6RUwG;@b5Y<c6E@xj0mV%Ccqe;9E+qI5%$K`p;hLS&>1I1XmOugL8epQ)qZnVDWUwj1DDIDTCHw(b5k7v9D}6qa&~gGiaI$N?M5{<3oqf#4NjpN?SzQve^f0B8372N}bo2LPki#jIcy9<k+xO4(5i)VFpTuMJPG+W?@3DIE-3;fZ{<9p%Rt`EUHp#D!DbGuUrWHC;|NPOr9d%cgcXKxVL?R-Y^{ow+YR3De<GZjm&g@wYH`DX~#rKwp2Sad5^jDk`uvDIW=5=m=CBIVO3#{ZaJi!7OGhjToH@;t;6*?CZ3Jd>fk;YdMjEQUSI0)+ER74sqZ+o44HxYNLY$sV?g9ppU(|+LY>czRH-(*u~p+Vs^*yDgwE_v0F=$>%u{BHNo;LwrD4nvY$N2Z8XaC=syMDxaa<uPr&LyFe#CrfNbkpPphKf-oT;$Jna1;88qa%aEbpZR@0Be`VD4=Nj||?j3$HI#J721Hendp?OJP+x?xjRQ_8DDh#w+VU3N5-MPlpwyFX{FsDRr^3j^@df;K`UeQliDO1C6Odwo%HHsgx&EIZtlNUMOb(bWKvcqh)6$t|iQhB}@1-C;4-~s}jr=rkOU55cJuk(1khFoCVNWgkb^%Q+Y3|xDG|biV}F<vY`<EV0oybJC)fDPVwg)`QS8AmOWmK>>2>Iq*FJC&(vqb&1v8piAH-YdmPDz`#oVovIGat$|upvCk3Jc?acbfrG%Kz6=*kNHdr!My^TjBqP5UWnGjg<J?4X{B8k$MT9>}m6h+mI<jxX1Np-@XQwbf#<nl40mK}*4{3uU6aQ$n{lSLAo0;jo!gY#LY#9G2W2kq?FtU=K*f;sibi9*4_0LCC3k*zSL<{{EJU6dupe*wzNP!=a3==q+M1=l>lKp|r=$*j{FN{7iE3MI@(&vF@_xNJEQTH6SM=E_(($ux6iSUik0^Fy3VW96D~Ck%v@oJ$=FCZf1noWQD<`2ZmkJz`v>kFZ+sEFu@Lsi{u|;`Gl-zN=xGY8s>AuLWF=QwgA1=ZQ-bIe9pdvqUnSpV?eV&X#nuk=PsqNR@mniv){iXmK7V7w2(uI*(Ix#j=|rrnbfFwJwy2_l=W{T!o!BQi4&1Yp98p$M#uA-g`K0vVr1&z-P(Bd6#IAxXLEI@^~yMY;5XCtb}RtXyVGFi7O8iSJ_lSybw%w1LbR?&Xo{XE+wv9OkBB`xN@4fvfKgwhqf9$w~;QCAu0}mYrlpx-oag|VXfeRXdV!FRnQ<wKeV`ZoPrLJ+B1V@M{B7at)+wL6)|Ehwd$~o@RhXokz7d7&b+HaEw#m3YKyhh7JnDuxr-@(hHZ1iQJ*x(!K={y#uM=NDI6?UUfs#L#MFxgi@3c=rdL%xhsPva@`}F|8T|;p;)NpW5oO~>_U;WYFO7R#sjj(FXwRv%*>j_uDC~5SLgppqvq8)pj8h@5<t(l}!&-cK5jhXj*8;&Z_R3?~E640rNWY|8z)6?g!9pbDqpGnpONB4hx^nc&5qiDg9hIV29#2wv%w_~U3`WljMy?W`uU{Y#LBe{Q@cQtbW-@8Gl!<c0M3X)L_v~*|`#)(v{`AAI*Wb%I%k%B}KXi}$`L7iWoI5@LwQ`R1bpGpqqkYkz>8f6jLRR<t-9z_5j%Aw7`hQ#Pw)eY3cTmTheb>Dl65Idnhwj7i@YQTq&((MB?Qy5R{rJA!-v0fd-rl#Xn+|?frM}*6w-Pcvs$Xlh-+erGZ`9BKczdOd72C9Emg~*3-7KvnOIk!mdhPwHgX!O_*>0Bg+KGcto6Q<)TitnDo3*Jr&&w75Suf@@Jy$<`+~fCU7f(6Gy4yeWht6L0r!()j-CBKnef`aszrFqC`X@R4Y5%D1+Q)8x`Y2{otsCEBwe4%t^rmxu)d>A%vvk&*m09|B*{l&%xwb-RytJNH%hoR8Agwld6q>7FZn}XpbbtT+sNJn<@9q@WYQ6j8!`-f3<FB7itWw?m?EwOZ-gvin_cyJotNz5r<IP`h;Q3lCL|6F7=cB9C0mHs+=j!!TU)=iRU4LNPb63B@nAc0vhN)_+rPrIr(0YxflH9uL;7_;94e?qS^vX7-t~zbo+B3p4T+pmcjgHQwoA*0B!eg%)hnt-oiFQ!kr*?I`dsGYU=ibT=)p%{PORZnLoZTf)C1cY^(sT9psCN6=vb)>;fuCxR>iHO53?k$4AE0~vS!|zf4@WpwYVq^+yLake|9tnIqQ%QG^)jGxdRtwqn;n#)nn54?|EpEIfq#(F3;DIfakqMphgjl9>h}1iALHkay>w^xtXj+NrhT~G?a}?~hud|BSt8f$yOlgee<IxC(|xzU-O9{>NnP}BZPCJe*x+{#Uw(pOEZ*QM^zHe-X7Og$UnPfeFXm4(^1eMB<Oh#^->)#bM|J4B^+A~{&FV+^mdD-Iq1|-qxi5Xk@PwP)zMEnCy>EX<_vY&Rox1CwUHksL&lla2gVXQ#w+~OJ8ban-9=&PRUVNNw_o7tu$sw59)TTDIsZDKaQ=8h<rZ%;yO>JsZo7&W-Hnpiu?SHBL3G;-TuK*YU00"""

output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "x036_public_output"
output_dir.mkdir(parents=True, exist_ok=True)
archive_path = output_dir / "x036_conditional_milk_delay.tar.gz"
archive_path.write_bytes(base64.b85decode(PAYLOAD_B85.encode("ascii")))
assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == ARCHIVE_SHA256

with tarfile.open(archive_path, "r:gz") as bundle:
    names = bundle.getnames()
    assert set(names) == set(EXPECTED_FILES)
    assert all(not Path(name).is_absolute() and ".." not in Path(name).parts for name in names)
    bundle.extractall(output_dir, filter="data")

for name, expected_hash in EXPECTED_FILES.items():
    actual_hash = hashlib.sha256((output_dir / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, (name, actual_hash)

print("Clean source-identical X036 archive restored:", archive_path)
print("Archive SHA-256:", ARCHIVE_SHA256)
print("Verified files:", ", ".join(EXPECTED_FILES))


## X036-specific policy source

The two cells below display the small policy wrapper and the frozen ridge model. The downloadable archive also contains the attributed community route and the retry layer unchanged.


In [ ]:
from IPython.display import Code, Markdown, display

display(Code(filename=str(output_dir / "main.py"), language="python"))
display(Code(filename=str(output_dir / "x036_supply_model.py"), language="python"))
display(Markdown((output_dir / "PROVENANCE.md").read_text(encoding="utf-8")))


## Full-game package smoke test

This checks that the exact released entrypoint loads and completes a full season from both player seats against Kaggle's deterministic starter agent. Rewards here are only a packaging smoke test, not competitive evidence.


In [ ]:
from kaggle_environments import make

main_path = str(output_dir / "main.py")
results = []
for seat in (0, 1):
    agents = [main_path, "starter"] if seat == 0 else ["starter", main_path]
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 36036}, debug=False)
    env.run(agents)
    final = [(state.status, state.reward) for state in env.steps[-1]]
    assert final[seat][0] == "DONE"
    results.append({"x036_seat": seat, "final": final})

print(results)


## What to improve next

Treat X036 as an engineering and experimental baseline:

1. Keep the control policy frozen and change one decision at a time.
2. Evaluate paired seeds in both seats against several frozen opponents.
3. Measure win margin and failure modes, not only the agent's own final bank.
4. Separate offline model quality from policy value: a better forecast does not automatically produce a better action.
5. Use fresh holdout seeds before promoting a change, and keep public/runtime features leakage-free.

The exact `x036_conditional_milk_delay.tar.gz` produced by this notebook can be downloaded from the notebook output and submitted directly.
